# RTStream V2 Cookbook

Build a live video workflow in small, inspectable recipes:

**connect → realtime → transcribe (optional) → understand → read records → index → search → alert → clean up → export (optional)**

This notebook uses the public VideoDB SDK only. Run the core cells from top to bottom, then choose the optional recipes you need. It is designed for **dev** by default.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/indexing-v2/rtstream/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## The mental model

| Object | ID | Purpose | Lifecycle |
|---|---|---|---|
| RTStream | `rts-*` | Connects a live source and optionally retains its recording | `stop()` |
| Understanding | `und-*` | Runs an analyzer on every time window | `start()` / `stop()` |
| Index | `idx-*` | Materializes one understanding output for records and search | `start()` / `stop()` |
| Alert | server-generated ID | Evaluates an event against an index | `enable_alert()` / `disable_alert()` |

Understanding and indexing are separate first-class jobs. An understanding produces named outputs; an index consumes one output descriptor such as `understanding.outputs["scene"]`.

> **Cost and cleanup:** a live RTStream keeps workers running. If any recipe fails, jump to **Cleanup** and run it.

## 1. Install the current V2 SDK

The V2 understanding/index surface currently lives on the SDK feature branch. The WebSocket extra supports live understanding, transcript, and alert messages. Restart the kernel if Jupyter asks you to after installation.

In [ ]:
!pip install -q --force-reinstall --no-cache-dir "git+https://github.com/video-db/videodb-python.git@feat/add-indexing-v2" python-dotenv websockets ipywidgets

## 2. Imports, shared state, and safe helpers

The `_rts` dictionary keeps every resource in one place, so later cells can be rerun and the cleanup recipe can always find what was created.

In [ ]:
import asyncio
import json
import os
import threading
import time
from datetime import datetime, timezone
from getpass import getpass

import videodb
import ipywidgets as widgets
from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display

load_dotenv()
_rts = globals().get("_rts", {})
_rts.setdefault("messages", [])

def records_from(payload):
    """Normalize the record envelopes used by understanding, indexes, and transcripts."""
    if isinstance(payload, list):
        return payload
    if not isinstance(payload, dict):
        return []
    for key in ("records", "scene_index_records", "scenes", "transcription_records"):
        value = payload.get(key)
        if isinstance(value, list):
            return value
    return []

def poll_records(fetch, timeout=180, interval=10, label="records"):
    """Poll a read call until it returns records; fresh indexes may briefly return 404."""
    deadline = time.monotonic() + timeout
    last_error = None
    while time.monotonic() < deadline:
        try:
            payload = fetch()
            rows = records_from(payload)
            print(f"{label}: {len(rows)}", end="\r")
            if rows:
                print(f"{label}: {len(rows)}")
                return payload, rows
        except Exception as exc:
            last_error = exc
            print(f"{label}: waiting ({type(exc).__name__})", end="\r")
        time.sleep(interval)
    print()
    if last_error:
        print(f"No {label} before timeout; last read error: {last_error}")
    else:
        print(f"No {label} before timeout. The live pipeline may still be warming up.")
    return {}, []

def show_rows(rows, limit=10):
    for row in rows[:limit]:
        if not isinstance(row, dict):
            print(row)
            continue
        data = row.get("data") or {}
        text = (data.get("scene_description") if isinstance(data, dict) else None)
        text = text or row.get("text") or row.get("description") or data
        print(f"[{row.get('start')} → {row.get('end')}] {text}")

def channel_messages(channel):
    return [m for m in _rts["messages"] if isinstance(m, dict) and m.get("channel") == channel]

print("Helpers ready.")

## 3. Configure the cookbook

Choose a sample source or supply your own RTSP URL. The two `store` switches are independent:

- **Stream storage** retains the recording and enables the export recipe after the stream stops.
- **Understanding storage** retains analyzer output so it can be read and indexed. Keep this on for the full cookbook.

Set `VIDEO_DB_API_KEY` (or `STAGING_API_KEY`) in your environment. If neither is present, the connect cell asks for it without saving it in the notebook.

In [ ]:
source_picker = widgets.Dropdown(
    options=[
        ("Baby in a crib", "rtsp://samples.rts.videodb.io:8554/crib"),
        ("Cricket match", "rtsp://samples.rts.videodb.io:8554/cricket"),
        ("Custom RTSP URL", "custom"),
    ],
    description="Source",
)
custom_url = widgets.Text(description="Custom URL", placeholder="rtsp://host:8554/feed")
base_url = widgets.Dropdown(
    options=[("Dev", "https://api.dev.videodb.io"), ("Production", "https://api.videodb.io")],
    description="API",
)
window = widgets.Dropdown(options=["5s", "10s", "15s", "30s"], value="10s", description="Window")
frame_count = widgets.IntSlider(value=5, min=1, max=12, description="Frames")
prompt = widgets.Textarea(value="Describe the scene clearly and mention important people, objects, and actions.", description="Prompt", layout=widgets.Layout(width="800px"))
stream_store = widgets.Checkbox(value=False, description="Retain stream recording")
understanding_store = widgets.Checkbox(value=True, description="Persist understanding output")
include_audio = widgets.Checkbox(value=False, description="Request audio (for transcription recipe)")
display(widgets.VBox([base_url, source_picker, custom_url, window, frame_count, prompt, stream_store, understanding_store, include_audio]))

In [ ]:
stream_url = custom_url.value.strip() if source_picker.value == "custom" else source_picker.value
if not stream_url:
    raise ValueError("Enter a custom RTSP URL or choose a sample source.")

CONFIG = {
    "base_url": base_url.value,
    "stream_url": stream_url,
    "window": window.value,
    "frame_count": frame_count.value,
    "prompt": prompt.value.strip(),
    "stream_store": stream_store.value,
    "understanding_store": understanding_store.value,
    "include_audio": include_audio.value,
}
display(JSON(CONFIG))

## 4. Connect to VideoDB

In [ ]:
environment_key = "PROD_API_KEY" if CONFIG["base_url"] == "https://api.videodb.io" else "STAGING_API_KEY"
api_key = os.getenv("VIDEO_DB_API_KEY") or os.getenv(environment_key)
if not api_key:
    api_key = getpass("VideoDB API key: ")
if not api_key:
    raise RuntimeError("A VideoDB API key is required.")

connection = videodb.connect(api_key=api_key, base_url=CONFIG["base_url"])
collection = connection.get_collection()
_rts.update({"connection": connection, "collection": collection})
print("Connected to collection:", collection.id)

## 5. Connect the live stream

This is a pull stream: VideoDB reads from the RTSP source. Video is enough for understanding. Enable audio in the configuration only if the source actually has an audio track and you plan to run the transcription recipe.

In [ ]:
media_types = ["video", "audio"] if CONFIG["include_audio"] else ["video"]
run_started_at = time.time()
_rts["started_at"] = run_started_at
rtstream = collection.connect_rtstream(
    url=CONFIG["stream_url"],
    name=f"rtstream-v2-cookbook-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}",
    media_types=media_types,
    store=CONFIG["stream_store"],
)
_rts["rtstream"] = rtstream
print("stream id:", rtstream.id)
print("status:", rtstream.status)
print("media types:", media_types)

## 6. Open the realtime channel

The SDK hides the WebSocket URL and connection handshake. The returned `connection_id` is attached to understanding, transcription, and alert operations. Live messages and durable records are complementary: the socket is for immediate updates; `get_records()` is for reading persisted output later.

In [ ]:
def start_realtime_listener(connection, collection_id, timeout=25):
    ready = threading.Event()
    state = {"messages": [], "error": None}

    def runner():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        state["loop"] = loop

        async def listen():
            ws = connection.connect_websocket(collection_id)
            state["socket"] = ws
            await ws.connect()
            state["connection_id"] = ws.connection_id
            ready.set()
            async for message in ws.receive():
                state["messages"].append(message)

        try:
            loop.run_until_complete(listen())
        except Exception as exc:
            state["error"] = f"{type(exc).__name__}: {exc}"
            ready.set()
        finally:
            loop.close()

    thread = threading.Thread(target=runner, name="rtstream-v2-ws", daemon=True)
    state["thread"] = thread
    thread.start()
    ready.wait(timeout)
    return state

ws_state = start_realtime_listener(connection, collection.id)
_rts["ws_state"] = ws_state
_rts["messages"] = ws_state["messages"]
ws_connection_id = ws_state.get("connection_id")
if not ws_connection_id:
    print("Realtime connection was not established:", ws_state.get("error"))
    print("You can continue without realtime; durable reads still work.")
else:
    print("realtime connection id:", ws_connection_id)

## 7. Start an understanding

Time segmentation creates one analysis window every `window` interval. The VLM analyzer samples a fixed number of frames from each window and produces a named output called `scene`.

A successful call returns an independent `und-*` resource. It is not a legacy scene index.

In [ ]:
understanding = rtstream.understand(
    segmentation={"type": "time", "window": CONFIG["window"]},
    analyzers=[
        {
            "type": "vlm",
            "name": "scene",
            "sampling": {"frame_count": CONFIG["frame_count"]},
            "config": {"prompt": CONFIG["prompt"]},
        }
    ],
    store=CONFIG["understanding_store"],
    ws_connection_id=ws_connection_id,
)
_rts["understanding"] = understanding
print("understanding id:", understanding.id)
print("status:", understanding.status)
print("outputs:")
display(JSON(understanding.outputs))

### Inspect the first-class resource

`get_understanding()` and `list_understanding()` manage `und-*` jobs. The assertion below documents the important V2 boundary: an understanding does not appear in the legacy scene-index list.

In [ ]:
fetched_understanding = rtstream.get_understanding(understanding.id)
understanding_ids = [item.id for item in rtstream.list_understanding()]
legacy_scene_ids = [getattr(item, "id", None) or getattr(item, "rtstream_index_id", None) for item in rtstream.list_scene_indexes()]

assert fetched_understanding.id == understanding.id
assert understanding.id in understanding_ids
assert understanding.id not in legacy_scene_ids
print("understanding is gettable, listable, and independent of legacy scene indexes ✅")

## 8. Observe live understanding messages

Understanding realtime messages use channel `visual_index`. The `data.index_id` field carries the `und-*` ID for compatibility with the current message envelope. Give the first VLM window time to complete.

In [ ]:
deadline = time.monotonic() + 120
while time.monotonic() < deadline and not channel_messages("visual_index"):
    print(f"waiting for live understanding output ({len(_rts['messages'])} total messages)", end="\r")
    time.sleep(5)
print()
visual_messages = channel_messages("visual_index")
print("visual_index messages:", len(visual_messages))
if visual_messages:
    display(JSON(visual_messages[-1]))
elif not ws_connection_id:
    print("Skipped: realtime was not connected.")
else:
    print("No live message yet. Continue to the durable read and try this cell again later.")

## 9. Read durable understanding records

With understanding storage enabled, `get_records(start, end)` returns structured output for a Unix-timestamp range. The first record can take a few windows because the worker and analyzer need to warm up.

> If you selected **Persist understanding output = off**, this recipe is expected to return no durable records.

In [ ]:
range_start = time.time() - 15 * 60
understanding_payload, understanding_records = poll_records(
    lambda: understanding.get_records(start=range_start, end=time.time(), output="scene", page_size=100),
    timeout=180,
    interval=10,
    label="understanding records",
)
_rts["understanding_records"] = understanding_records
show_rows(understanding_records)

## 10. Build an index from the understanding

An index consumes a named output descriptor—not the whole understanding. `use_for=["semantic"]` makes the materialized records searchable. Multiple indexes can consume the same understanding output with different names or capabilities.

In [ ]:
if not CONFIG["understanding_store"]:
    raise RuntimeError("Turn on 'Persist understanding output', then recreate the stream and understanding before indexing.")
if "scene" not in understanding.outputs:
    raise RuntimeError(f"The understanding has no 'scene' output: {understanding.outputs}")

index = rtstream.index(
    source=understanding.outputs["scene"],
    name="rtstream-v2-cookbook-index",
    use_for=["semantic"],
)
_rts["index"] = index
print("index id:", index.id)
print("status:", index.status)
print("source understanding:", index.source_understanding_id)
print("capabilities:", index.use_for)

### Inspect the first-class index resource

`get_index()` reads one `idx-*` resource and `list_indexes()` returns every V2 index attached to the stream. These are separate from `list_scene_indexes()`, which exists only for the legacy scene-index API.

In [ ]:
fetched_index = rtstream.get_index(index.id)
index_ids = [item.id for item in rtstream.list_indexes()]
legacy_scene_ids = [getattr(item, "id", None) or getattr(item, "rtstream_index_id", None) for item in rtstream.list_scene_indexes()]

assert fetched_index.id == index.id
assert index.id in index_ids
assert index.id not in legacy_scene_ids
print("index is gettable, listable, and independent of legacy scene indexes ✅")

### Read materialized index records

These are the searchable derivatives of the understanding output. V2 records are delivered by the Indexer and served through the RTStream read API; a fresh index can briefly return no records while the first materialization completes.

In [ ]:
index_payload, index_records = poll_records(
    lambda: index.get_records(start=range_start, end=time.time(), page=1, page_size=100),
    timeout=180,
    interval=10,
    label="index records",
)
_rts["index_records"] = index_records
show_rows(index_records)

## 11. Search the live index

Search is most useful after index records appear. Change the query and rerun this cell. Each result is an `RTStreamShot` with timestamps and text; `shot.generate_stream()` can create a playable clip for its time range.

In [ ]:
SEARCH_QUERY = "a person or an important action"

try:
    search_result = rtstream.search(query=SEARCH_QUERY, index_id=index.id, result_threshold=5)
    shots = search_result.get_shots()
    _rts["shots"] = shots
    print(f"{len(shots)} result(s) for: {SEARCH_QUERY!r}")
    for shot in shots:
        print(f"[{shot.start} → {shot.end}] score={shot.search_score} | {shot.text}")
except Exception as exc:
    print("Search is not ready yet:", exc)

## 12. Optional: create a live alert

An event describes what to detect; an alert attaches that event to this index. Set `RTSTREAM_ALERT_CALLBACK_URL` to your webhook URL before running. When the condition matches, delivery can arrive through both the callback and the collection WebSocket.

This recipe deliberately skips creation when no callback is configured, so a shared notebook never sends data to a surprise endpoint.

In [ ]:
EVENT_PROMPT = "A person is visible in the scene"
EVENT_LABEL = "person-visible"
CALLBACK_URL = os.getenv("RTSTREAM_ALERT_CALLBACK_URL", "").strip()

if not CALLBACK_URL:
    print("Skipped. Set RTSTREAM_ALERT_CALLBACK_URL and rerun this cell to create an alert.")
else:
    event_id = connection.create_event(event_prompt=EVENT_PROMPT, label=EVENT_LABEL)
    alert_id = index.create_alert(
        event_id=event_id,
        callback_url=CALLBACK_URL,
        ws_connection_id=ws_connection_id,
    )
    _rts.update({"event_id": event_id, "alert_id": alert_id})
    print("event id:", event_id)
    print("alert id:", alert_id)
    display(JSON(index.list_alerts()))

In [ ]:
# Run after creating an alert; rerun whenever you want the latest live-message summary.
print("all realtime messages:", len(_rts["messages"]))
print("understanding messages:", len(channel_messages("visual_index")))
print("alert messages:", len(channel_messages("alert")))
if channel_messages("alert"):
    display(JSON(channel_messages("alert")[-1]))

## 13. Optional: inspect alert playback and generate a clip

A fired alert can already contain `player_url` and `stream_url`. Those URLs point to the alert window prepared by the realtime pipeline. You can also call `rtstream.generate_stream(start, end)` for the same timestamps to request a fresh playable clip.

Run this after at least one `alert` message arrives. If nothing has fired, the cell explains that there is no window to clip.

In [ ]:
alert_messages = channel_messages("alert")
if not alert_messages:
    print("No alert has fired yet. Leave the stream running and try this cell again.")
else:
    latest_alert = alert_messages[-1]
    alert_data = latest_alert.get("data") or latest_alert
    print("label:", alert_data.get("label"))
    print("window:", alert_data.get("start"), "→", alert_data.get("end"))
    print("alert player URL:", alert_data.get("player_url"))
    print("alert stream URL:", alert_data.get("stream_url"))

    if alert_data.get("start") is None or alert_data.get("end") is None:
        print("This alert has no complete time window, so a clip cannot be generated.")
    else:
        generated_clip_url = rtstream.generate_stream(
            start=int(alert_data["start"]),
            end=int(alert_data["end"]),
        )
        _rts["generated_alert_clip"] = generated_clip_url
        print("generated clip URL:", generated_clip_url)

## 14. Optional: transcription

Run this only when the source has audio and **Request audio** was enabled before stream creation. The same WebSocket connection receives live transcript messages; `get_transcript()` reads durable final segments.

In [ ]:
if not CONFIG["include_audio"]:
    print("Skipped. Recreate the stream with 'Request audio' enabled to run transcription.")
else:
    response = rtstream.start_transcript(ws_connection_id=ws_connection_id, engine="assemblyai")
    _rts["transcription_started"] = True
    display(JSON(response))
    transcript_payload, transcript_records = poll_records(
        lambda: rtstream.get_transcript(page_size=1000, engine="assemblyai"),
        timeout=120,
        interval=10,
        label="transcript records",
    )
    _rts["transcript_records"] = transcript_records
    show_rows(transcript_records)

## 15. Optional: pause and resume processing jobs

Stopping an understanding or index pauses that job without stopping the parent stream. Starting it resumes processing new windows. This cell is opt-in to avoid disrupting a live demo accidentally.

In [ ]:
RUN_PAUSE_RESUME_DEMO = False

if not RUN_PAUSE_RESUME_DEMO:
    print("Set RUN_PAUSE_RESUME_DEMO=True and rerun to exercise both lifecycles.")
else:
    index.stop()
    understanding.stop()
    print("stopped:", rtstream.get_index(index.id).status, rtstream.get_understanding(understanding.id).status)
    understanding.start()
    index.start()
    print("resumed:", rtstream.get_index(index.id).status, rtstream.get_understanding(understanding.id).status)

## 16. Cleanup — always run this

Cleanup order matters: stop new alert/transcript activity, stop derived jobs, stop the stream, then close the socket. The helper is best-effort and idempotent, so it is safe to rerun after a partial failure.

In [ ]:
def cleanup_rtstream_cookbook():
    results = {}

    index_obj = _rts.get("index")
    alert_id = _rts.get("alert_id")
    if index_obj is not None and alert_id:
        try:
            index_obj.disable_alert(alert_id)
            results["alert"] = "disabled"
        except Exception as exc:
            results["alert"] = f"disable failed: {exc}"

    stream_obj = _rts.get("rtstream")
    if stream_obj is not None and _rts.get("transcription_started"):
        try:
            stream_obj.stop_transcript(engine="assemblyai")
            results["transcription"] = "stopped"
        except Exception as exc:
            results["transcription"] = f"stop failed: {exc}"

    for key in ("index", "understanding", "rtstream"):
        obj = _rts.get(key)
        if obj is None:
            continue
        try:
            obj.stop()
            results[key] = "stopped"
        except Exception as exc:
            results[key] = f"stop failed: {exc}"

    ws = _rts.get("ws_state", {})
    loop, socket = ws.get("loop"), ws.get("socket")
    if loop and socket and loop.is_running():
        try:
            asyncio.run_coroutine_threadsafe(socket.close(), loop).result(timeout=10)
            results["websocket"] = "closed"
        except Exception as exc:
            results["websocket"] = f"close failed: {exc}"

    return results

cleanup_result = cleanup_rtstream_cookbook()
display(JSON(cleanup_result))

## 17. Verify durable reads after stopping

Stopping compute should not erase stored product data. This recipe reads every relevant public surface again after cleanup:

- stream status from the server,
- list/get Understanding and its timestamp-ranged records,
- list/get Index and its timestamp-ranged records,
- legacy scene-index separation, alert state, and transcript records.

Each read is independent. One failure is reported without hiding the remaining results.

In [ ]:
def resource_summary(resource):
    return {
        "id": getattr(resource, "id", None),
        "status": getattr(resource, "status", None),
        "name": getattr(resource, "name", None),
    }

def read_safely(results, label, fetch):
    try:
        results[label] = fetch()
        print(f"{label}: OK")
    except Exception as exc:
        results[label] = {"error": f"{type(exc).__name__}: {exc}"}
        print(f"{label}: {results[label]['error']}")

post_stop_end = time.time()
post_stop_reads = {}
read_safely(post_stop_reads, "stream", lambda: resource_summary(collection.get_rtstream(rtstream.id)))
read_safely(post_stop_reads, "understandings", lambda: [resource_summary(item) for item in rtstream.list_understanding()])
read_safely(post_stop_reads, "understanding", lambda: resource_summary(rtstream.get_understanding(understanding.id)))
read_safely(post_stop_reads, "understanding_records", lambda: understanding.get_records(start=run_started_at - 60, end=post_stop_end, output="scene", page_size=100))
read_safely(post_stop_reads, "indexes", lambda: [resource_summary(item) for item in rtstream.list_indexes()])
read_safely(post_stop_reads, "index", lambda: resource_summary(rtstream.get_index(index.id)))
read_safely(post_stop_reads, "index_records", lambda: index.get_records(start=run_started_at - 60, end=post_stop_end, page_size=100))
read_safely(post_stop_reads, "legacy_scene_indexes", lambda: [getattr(item, "id", None) or getattr(item, "rtstream_index_id", None) for item in rtstream.list_scene_indexes()])
read_safely(post_stop_reads, "alerts", lambda: index.list_alerts())
read_safely(post_stop_reads, "transcript", lambda: rtstream.get_transcript(start=run_started_at - 60, end=post_stop_end, page_size=100))
_rts["post_stop_reads"] = post_stop_reads
display(JSON(post_stop_reads))

## 18. Optional: export the retained recording

Export is available only when **Retain stream recording** was enabled before stream creation. The stream must be stopped first. Recording finalization is asynchronous, so this recipe retries briefly instead of treating the first not-ready response as a recording failure. Export is idempotent: rerunning returns the same asset.

In [ ]:
if not CONFIG["stream_store"]:
    print("Skipped. The stream was created with recording storage disabled.")
else:
    EXPORT_RETRIES = 6
    export_result = None
    export_error = None
    for attempt in range(1, EXPORT_RETRIES + 1):
        try:
            export_result = rtstream.export(name="RTStream V2 Cookbook Recording")
            break
        except Exception as exc:
            export_error = exc
            print(f"export attempt {attempt}/{EXPORT_RETRIES}: recording is not ready ({exc})")
            time.sleep(5)

    if export_result is None:
        print("Recording did not finalize during the retry window:", export_error)
    else:
        _rts["export"] = export_result
        print("video id:", export_result.video_id)
        print("duration:", export_result.duration)
        print("stream URL:", export_result.stream_url)
        print("player URL:", export_result.player_url)

## Cookbook recap

You now have the complete RTStream V2 progression:

1. Connect a pull RTStream (`rts-*`).
2. Open one collection realtime connection.
3. Run a first-class understanding (`und-*`) over time windows.
4. Observe live `visual_index` messages and read durable understanding records.
5. Materialize one named output into a first-class index (`idx-*`).
6. Read index records and run semantic search.
7. Optionally attach an event alert, inspect its playback URLs, generate a clip, and enable transcription.
8. Stop every resource and verify that durable reads still work.
9. Optionally export a retained recording after asynchronous finalization.

The key V2 idea is composition: **stream → understanding output → index → alert/search**. Each resource has its own lifecycle, while the public SDK keeps the underlying Conductor, Manager, Fetcher, Indexer, storage, and realtime services behind the API boundary.